# 12b · Frozen P6 subsets for the explanations (CPU)

**CPU runtime** (no GPU; about 0.08 compute units per hour; about 0.05 CU in all, estimated). No model runs here.

**Decisions (Dhanush, 18 Nov 2026):** P6 keeps the spec's mix and is frozen by the SHA-256 of its sorted uid list **before any model is scored on it**:
- **400 DiffSeg30k held-out** (Glide/Kolors edits; `test_P2` outside the CP4 dev split; it may overlap the P1/P2 test subsets);
- **400 SID-Set validation tampered** (masks binarized at > 127 when scored);
- **400 BR-Gen** from the frozen BR-Gen subset, methods other than SDXL;
- **150 real, about 50 from each of:**
  - GenImage P1 real, outside the dev split and the frozen P1 subset;
  - DiffSeg30k validation unedited (topped up from SID-Set validation real when short);
  - Chameleon real, outside P4;
  - each one checked against the training pool: no exact hash match and pHash distance > 6;
- **150 fully generated** from P1 outside the dev split, spread evenly over the 8 generators.

**Nested subsets:** `P6_mllm` (500, stratified by part) and `P6_deletion` (300, edited images only, from the three edited parts).

**Steps:** the Chameleon zip is unpacked again to the local disk (the real images outside P4 are ranked first; only the first 400 are hashed). The 400 chosen BR-Gen images and their masks are extracted with 7z, opened and hashed, so notebook 14 can check it reads the same files.

**It stops and asks Dhanush** if:
- an input is missing, or `eval_subsets.json` / `brgen_subset.json` was changed;
- a P6 part is short;
- a BR-Gen file fails to open or its mask size differs;
- `eval_subsets.json` already holds different P6 subsets.

**Results** (send to Claude):
- `Paper 1/runs/checkpoints/eval_subsets.json` (new keys `P6`, `P6_mllm`, `P6_deletion`, `P6_brgen_files`; P1 and P4 untouched);
- `Paper 1/runs/checkpoints/p6_subsets_record.json` (counts per part and source, skipped near-duplicates, compute).

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings and pre-flight
import datetime, gzip, csv, json, logging, shutil, subprocess, tempfile, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
from eviground.data import brgen, chameleon, hashing
from eviground.env import get_paths, session_info
from eviground.eval import devsplit, subsets
from eviground.utils import session

SEED = 42
CU_PER_HOUR = 0.08         # CPU runtime: read the live rate in Colab's resource panel
CU_LEFT = None             # optional: the balance shown in Colab; the pre-flight stops if EST_CU exceeds it
EST_CU = 0.05
CHAMELEON_HASHED = 400     # Chameleon real images (outside P4) hashed, in P6 rank order; 50 must pass
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_p6_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    EST_CU, CHAMELEON_HASHED = 0.0, 60
paths.ensure()
CK = paths.runs / 'checkpoints'
CP1_MANIFEST = CK / 'cp1' / 'manifest_cp1.csv.gz'
DEV_SPLIT = CK / 'cp4_dev_split.json'
SUBSETS_FILE = CK / 'eval_subsets.json'
BRGEN_SUBSET = CK / 'brgen_subset.json'
ZIP = paths.datasets / '05_Chameleon' / chameleon.ZIP_NAME
BRGEN_DIR = paths.datasets / '04_BR-Gen'
ARCHIVE = BRGEN_DIR / brgen.LAST_VOLUME
LOCAL = paths.local_data / 'chameleon'
if DRY:   # fake world: a manifest with every P6 pool, a dev split, P1/P4, a Chameleon zip, a BR-Gen archive
    import io, zipfile
    from PIL import Image

    def _png(i, size=16):
        buf = io.BytesIO()
        Image.new('RGB', (size, size), (i % 251, (7 * i) % 253, (13 * i) % 255)).save(buf, format='PNG')
        return buf.getvalue()

    rows = []
    for i in range(450):
        rows += [{'uid': f'diffseg30k/val/p2_{i}.png', 'source': 'diffseg30k', 'role': 'test_P2', 'label': 'edited'},
                 {'uid': f'sidset/val/t{i}.png', 'source': 'sidset', 'role': 'test_P3b', 'label': 'edited'},
                 {'uid': f'sidset/val/r{i}.png', 'source': 'sidset', 'role': 'test_P3b', 'label': 'real',
                  'sha256': f'sr{i}', 'phash': str(i)}]
    for g in ('adm', 'biggan', 'glide', 'midjourney', 'sd14', 'sd15', 'vqdm', 'wukong'):
        for lab in ('real', 'generated'):
            rows += [{'uid': f'genimage/{g}/{lab}/{i}.png', 'source': 'genimage', 'role': 'test_P1', 'split': 'test',
                      'generator': g, 'label': lab, 'sha256': f'{g}{lab}{i}'} for i in range(40)]
    rows += [{'uid': f'diffseg30k/val/u{i}.png', 'source': 'diffseg30k', 'role': 'unused_validation',
              'editors': '', 'sha256': f'du{i}'} for i in range(30)]   # 30 < 50: SID-Set real tops up
    rows += [{'uid': f'coco/train/{i}.jpg', 'source': 'coco', 'role': 'train_pool', 'label': 'real',
              'sha256': f'c{i}', 'phash': str(0x5555555555555555 ^ i)}
             for i in range(5)]
    rows[-1]['sha256'] = 'admreal1'   # one GenImage real test image duplicates a training image
    CP1_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(CP1_MANIFEST, 'wt', newline='') as fh:
        cols = sorted({k for r in rows for k in r})
        w = csv.DictWriter(fh, fieldnames=cols); w.writeheader(); w.writerows(rows)
    devsplit.save(devsplit.build_dev_split([r for r in rows if r['role'] in ('test_P1', 'test_P2')], fraction=0.2),
                  DEV_SPLIT)
    ZIP.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP, 'w') as zf:
        for i in range(80):
            zf.writestr(f'Chameleon/test/0_real/r{i}.png', _png(i))
        for i in range(20):
            zf.writestr(f'Chameleon/test/1_fake/f{i}.png', _png(1000 + i))
    cham_items = chameleon.items([f'Chameleon/test/0_real/r{i}.png' for i in range(80)]
                                 + [f'Chameleon/test/1_fake/f{i}.png' for i in range(20)])
    p1_rows = [r for r in rows if r['role'] == 'test_P1']
    subsets.save({'P1': subsets.p1_test_subset(p1_rows, devsplit.load(DEV_SPLIT)['dev_uids'], per_label=3),
                  'P4': subsets.p4_subset(cham_items, total=20)}, SUBSETS_FILE)
    forged, masks = [], {}
    BRGEN_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE, 'w') as zf:
        for src in brgen.SOURCES:
            for reg in brgen.REGIONS:
                for i in range(17):
                    name = f'{src.lower()}_{i:06d}_{reg.lower()}.png'
                    mask = f'BR-Gen/Mask/{reg}/{src}/{name}'
                    zf.writestr(mask, _png(i))
                    for m in brgen.METHODS:
                        f = f'BR-Gen/Forged/{m}/{reg}/{src}/{name}'
                        zf.writestr(f, _png(i + 3))
                        forged.append(f); masks[f] = mask
    BRGEN_SUBSET.write_text(json.dumps({'design': 'paired', 'items': sorted(forged), 'masks': masks,
                                        'sha256': devsplit.file_list_sha256(forged)}))
PREFLIGHT = session.preflight(
    {'notebook': '12b P6 subsets', 'parts': subsets.P6_PARTS, 'nested': {'P6_mllm': subsets.P6_MLLM,
     'P6_deletion': subsets.P6_DELETION}, 'subsets file': SUBSETS_FILE, 'unpack to (local disk)': LOCAL},
    paths={'CP1 manifest': CP1_MANIFEST, 'CP4 dev split': DEV_SPLIT, 'eval_subsets.json (P1, P4)': SUBSETS_FILE,
           'brgen_subset.json': BRGEN_SUBSET, 'Chameleon.zip (Drive shortcut)': ZIP, 'BR-Gen archive': ARCHIVE},
    est_cu=EST_CU, cu_left=CU_LEFT)

In [ ]:
# Cell 3: inputs: the manifest pools, the training index, the frozen P1/P4 and BR-Gen subsets (checked)
from eviground.data.cp1 import _read_csv_gz
MANIFEST = _read_csv_gz(CP1_MANIFEST)
SPLIT = devsplit.load(DEV_SPLIT)
STORED = json.loads(SUBSETS_FILE.read_text())
P1_SHA, P4_SHA = STORED.get('P1', {}).get('sha256'), STORED.get('P4', {}).get('sha256')
for k in ('P1', 'P4'):
    assert subsets.intact(STORED.get(k, {})), f'{SUBSETS_FILE}: {k} is missing or was changed: stop and ask Dhanush'
BR = json.loads(BRGEN_SUBSET.read_text())
assert devsplit.file_list_sha256(BR['items']) == BR['sha256'], 'brgen_subset.json was changed: stop and ask Dhanush'
POOLS = subsets.p6_pools(MANIFEST, SPLIT['dev_uids'], STORED['P1']['uids'])
TRAIN_ROWS = [r for r in MANIFEST if r.get('role') == 'train_pool']
INDEX = subsets.TrainingIndex(TRAIN_ROWS)
print(f'training pool {len(TRAIN_ROWS)} rows: {len(INDEX.exact)} exact hashes, {INDEX.phash.size} pHashes')
print('pools:', {k: len(v) for k, v in sorted(POOLS.items())})
# how CP1 recorded DiffSeg30k's unedited validation images (role unused_validation, empty editors)
DS_VAL = [r for r in MANIFEST if r.get('source') == 'diffseg30k' and r.get('role') == 'unused_validation']
print(f"DiffSeg30k unused_validation: {len(DS_VAL)} rows, {len(POOLS.get('diffseg_unedited', []))} unedited",
      '(SID-Set validation real tops up if under 50)')

In [ ]:
# Cell 4: Chameleon real outside P4: unpack again (local disk), hash the first images in P6 rank order
t = time.time()
LOCAL.mkdir(parents=True, exist_ok=True)
local_zip = LOCAL.parent / chameleon.ZIP_NAME
shutil.copyfile(ZIP, local_zip)
NAMES = chameleon.unpack(local_zip, LOCAL)
p4 = set(STORED['P4']['uids'])
REAL_OUT = [r for r in chameleon.items(NAMES) if r['label'] == 'real' and r['uid'] not in p4]
HEAD = subsets.chameleon_candidates(REAL_OUT, CHAMELEON_HASHED)
CHAM_ROWS = []
for r in HEAD:
    h = hashing.image_hashes((LOCAL / r['uid']).read_bytes())
    if 'error' in h:
        raise RuntimeError(f"Chameleon {r['uid']}: {h['error']}: stop and ask Dhanush")
    CHAM_ROWS.append({**r, 'sha256': h['sha256'], 'pixel_sha256': h.get('pixel_sha256'), 'phash': h.get('phash')})
CHAM_S = round(time.time() - t, 1)
print(f'Chameleon: {len(NAMES)} unpacked, {len(REAL_OUT)} real outside P4, {len(CHAM_ROWS)} hashed in {CHAM_S} s')

In [ ]:
# Cell 5: P6 and its nested subsets; then the 400 BR-Gen images and masks: extract, open, hash
NEW = subsets.p6_subset(POOLS, BR['items'], CHAM_ROWS, INDEX)
P6 = NEW['P6']
print('P6', P6['images'], 'images', P6['per_part'], '| real', P6['real_sources'],
      '| SID-Set top-up', P6['sidset_top_up'], '| skipped near training', P6['skipped_near_training'])
exe = brgen.seven_zip()
if exe is None and not DRY:
    subprocess.run(['apt-get', 'install', '-y', '-q', 'p7zip-full'], check=True, capture_output=True)
    exe = brgen.seven_zip()
assert exe, '7z is not available'
BR_PICKS = sorted(u for u in P6['uids'] if P6['part_of'][u] == 'brgen')
out_dir = paths.local_data / 'brgen_p6'
t = time.time()
brgen.extract(ARCHIVE, BR_PICKS + sorted({BR['masks'][p] for p in BR_PICKS}), out_dir, exe)
from PIL import Image
FILES, BAD = {}, []
for p in BR_PICKS:
    try:
        with Image.open(out_dir / p) as im, Image.open(out_dir / BR['masks'][p]) as mk:
            im.load(); mk.load()   # decode the pixels, not only the header
            if im.size != mk.size:
                BAD.append((p, f'image {im.size} vs mask {mk.size}'))
        FILES[p] = {'sha256': hashing.file_sha256((out_dir / p).read_bytes()),
                    'mask': BR['masks'][p], 'mask_sha256': hashing.file_sha256((out_dir / BR['masks'][p]).read_bytes())}
    except (OSError, SyntaxError, ValueError) as e:   # Pillow decode errors
        BAD.append((p, repr(e)))
BRGEN_S = round(time.time() - t, 1)
print(f'BR-Gen: {len(BR_PICKS)} images + masks extracted and opened in {BRGEN_S} s; problems {len(BAD)}')
if BAD:
    raise RuntimeError(f'BR-Gen files failed: {BAD[:5]}: stop and ask Dhanush')
NEW['P6_brgen_files'] = {'files': FILES, 'sha256': devsplit.file_list_sha256(sorted(FILES))}

In [ ]:
# Cell 6: freeze (before any scoring). Existing keys are left untouched; an existing P6 is kept only if identical.
KEYS = ('P6', 'P6_mllm', 'P6_deletion')
STORED = json.loads(SUBSETS_FILE.read_text())
if any(k in STORED for k in KEYS):
    same = all(subsets.intact(STORED.get(k, {})) and STORED[k]['sha256'] == NEW[k]['sha256'] for k in KEYS)
    if not same:
        raise RuntimeError(f'{SUBSETS_FILE} already holds different (or altered) P6 subsets: stop and ask Dhanush')
    print('P6 subsets already frozen and identical: kept')
else:
    STORED.update(NEW)
    STORED['P6_frozen_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
    subsets.save(STORED, SUBSETS_FILE)
    print('saved', SUBSETS_FILE)
for k in KEYS:
    print(k, NEW[k]['images'], NEW[k]['sha256'][:12], json.dumps(NEW[k].get('per_part')))

In [ ]:
# Cell 7: the run record (counts per part and source, near-duplicates skipped, compute)
RECORD = {
    'step': 'frozen P6 subsets', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED, 'session': info,
    'pools': {k: len(v) for k, v in sorted(POOLS.items())}, 'training_pool_rows': len(TRAIN_ROWS),
    'diffseg_unused_validation': len(DS_VAL),
    'chameleon': {'unpacked': len(NAMES), 'real_outside_p4': len(REAL_OUT), 'hashed': len(CHAM_ROWS),
                  'seconds': CHAM_S},
    'brgen': {'images': len(BR_PICKS), 'seconds': BRGEN_S, 'files_sha256': NEW['P6_brgen_files']['sha256']},
    **{k: {kk: vv for kk, vv in subsets.summary(NEW[k]).items() if kk != 'part_of'} for k in KEYS},
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}
RECORD['compute'] = session.finish(CK, T0, CU_PER_HOUR, name='compute_p6_subsets.json',
                                   extra={'git_commit': info['git_commit'], 'est_cu': EST_CU})
out = CK / 'p6_subsets_record.json'
out.write_text(json.dumps(RECORD, indent=1, default=str))
print('saved', out)
if DRY:
    assert P6['images'] == 1500 and P6['per_part'] == subsets.P6_PARTS, P6['per_part']
    assert P6['real_sources'] == {'genimage': 50, 'diffseg_unedited': 50, 'chameleon': 50}, P6['real_sources']
    assert P6['sidset_top_up'] == 20, P6['sidset_top_up']
    real = {u for u in P6['uids'] if P6['part_of'][u] == 'real'}
    assert not real & set(STORED['P1']['uids']) and 'genimage/adm/real/1.png' not in real
    assert not any('/SDXL/' in u for u in P6['uids'])
    assert NEW['P6_mllm']['images'] == 500 and NEW['P6_deletion']['images'] == 300
    assert (STORED['P1']['sha256'], STORED['P4']['sha256']) == (P1_SHA, P4_SHA)

In [ ]:
# Cell 8: release the runtime (last cell; everything above is saved on Drive)
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED